In [1]:
%%capture
!pip install transformers sentencepiece datasets accelerate

In [2]:
import os
import re
import pandas as pd
from pathlib import Path
from typing import List, Dict
from transformers import pipeline

# =============================
# CONFIG
# =============================
INPUT_CSV = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/qna_data/nestle_qna_pairs.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset"
MODEL_NAME = "google/flan-t5-small"  # keep small; can switch to 'flan-t5-base' if you have GPU

# Ensure output dir exists
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
FINAL_CSV = os.path.join(OUTPUT_DIR, "extended_finetune_dataset.csv")

# =============================
# LOAD DATA
# =============================
df = pd.read_csv(INPUT_CSV)
# normalize column names
df = df.rename(columns={"Question": "question", "Answer": "answer"})
df = df[["question", "answer"]].dropna()
print("📊 Original dataset size:", len(df))

# =============================
# LLM PIPELINE (deterministic)
# =============================
generator = pipeline(
    "text2text-generation",
    model=MODEL_NAME,
    device=-1,          # CPU (set to 0 if you have GPU)
)

GEN_KW = dict(
    max_new_tokens=96,  # keep short, tight answers
    num_beams=4,
    do_sample=False,    # deterministic -> less vagueness
    repetition_penalty=1.05
)

# =============================
# HELPERS
# =============================
NUM_PAT = re.compile(r"[-+]?\d[\d\s.,]*")

def extract_numbers(text: str) -> List[str]:
    """Return normalized numeric tokens found in text."""
    nums = NUM_PAT.findall(text or "")
    # normalize numbers to a comparable form (strip spaces)
    return [re.sub(r"\s+", "", n) for n in nums]

def contains_all_key_numbers(candidate: str, reference: str) -> bool:
    """Ensure all numbers in reference appear in candidate (loosely normalized)."""
    ref_nums = extract_numbers(reference)
    if not ref_nums:
        return True  # nothing numeric to enforce
    cand_norm = re.sub(r"\s+", "", candidate or "")
    return all(n in cand_norm for n in ref_nums)

def run(prompt: str, **gen_kw) -> str:
    """Safe single-call generation with cleanup."""
    out = generator(prompt, **gen_kw)[0]["generated_text"].strip()
    return out

def safe_answer(q: str, gold_a: str, instruction: str, **gen_kw) -> str:
    """
    Generate an answer variant that must preserve gold facts.
    Falls back to a minimal rewrite if key numbers go missing.
    """
    prompt = (
        "You are a precise financial assistant. "
        "Use ONLY the facts from the GIVEN FACTS. Do not invent numbers, dates, or entities. "
        "If the facts do not answer the question, say 'Not specified.'\n\n"
        f"GIVEN FACTS:\n{gold_a}\n\n"
        f"QUESTION:\n{q}\n\n"
        f"INSTRUCTION:\n{instruction}\n\n"
        "OUTPUT:\nA single, concise sentence that answers the question using the given facts."
    )
    ans = run(prompt, **gen_kw)
    # If answer lost key numbers, try a stricter rewrite
    if not contains_all_key_numbers(ans, gold_a):
        strict_prompt = (
            "Rewrite the following answer to STRICTLY preserve all numbers and units from the FACTS. "
            "Do not add any new figures. Keep one concise sentence.\n\n"
            f"FACTS:\n{gold_a}\n\n"
            f"CURRENT ANSWER:\n{ans}\n\n"
            "REWRITE:"
        )
        ans2 = run(strict_prompt, **gen_kw)
        if contains_all_key_numbers(ans2, gold_a):
            return ans2
        # Fallback: use a trimmed, safe version of the original gold answer
        fallback_prompt = (
            "Convert the following FACTS into one concise sentence answer. "
            "Do not add any information not present in the text.\n\n"
            f"FACTS:\n{gold_a}\n\n"
            "ONE-SENTENCE ANSWER:"
        )
        ans3 = run(fallback_prompt, **gen_kw)
        return ans3.strip()
    return ans

def paraphrase_questions(q: str, gold_a: str, n: int = 3) -> List[str]:
    """
    Generate n paraphrases that keep semantics and require the same factual answer.
    Each paraphrase should be a single question, no list formatting.
    """
    prompt = (
        "Paraphrase the following question into distinct variants that ask for the SAME factual information. "
        "Keep each variant a single question, concise, and unambiguous. "
        "Do NOT change the meaning. Do NOT add numbers. Output each on a new line.\n\n"
        f"ORIGINAL QUESTION:\n{q}\n"
        f"KNOWN FACTUAL ANSWER (for your understanding only):\n{gold_a}\n\n"
        f"GENERATE {n} VARIANTS:"
    )
    text = run(prompt, **GEN_KW)
    # Split into lines and clean
    lines = [ln.strip("-• \t") for ln in text.splitlines() if ln.strip()]
    # Keep first n questions
    out = []
    for ln in lines:
        if ln.endswith("?"):
            out.append(ln)
        else:
            # ensure it's phrased as a question
            out.append(ln.rstrip(".") + "?")
        if len(out) == n:
            break
    # If fewer than n, pad by repeating original with minor reword
    while len(out) < n:
        out.append(q)
    return out

def analytical_question(q: str, gold_a: str) -> str:
    """
    Generate one analytical/contextual question that naturally follows from the same fact.
    """
    prompt = (
        "Create ONE analytical follow-up question based on the following Q/A. "
        "The follow-up should be natural, useful for an analyst, and grounded in the same topic. "
        "Do not ask for information not present in the known facts.\n\n"
        f"QUESTION:\n{q}\n\n"
        f"FACTS (answer):\n{gold_a}\n\n"
        "FOLLOW-UP QUESTION (one line):"
    )
    out = run(prompt, **GEN_KW).strip()
    if not out.endswith("?"):
        out = out.rstrip(".") + "?"
    return out

# =============================
# AUGMENTATION
# =============================
def enrich_entry(q: str, a: str) -> List[Dict[str, str]]:
    """
    Produce a diversified, high-quality set of Q/A pairs:
      - 4 stylistic answers (analyst, reasoning, short, eli5) that keep facts
      - 3 paraphrased questions with concise factual answers
      - 1 analytical follow-up + answer
    All answers are constrained to the original facts (no hallucinations).
    """
    records: List[Dict[str, str]] = []

    # Stylistic rewrites (answers only; preserve numbers)
    analyst_ans = safe_answer(q, a, "Professional analyst tone; crisp and factual.")
    records.append({"style": "analyst", "question": q, "answer": analyst_ans})

    reasoning_ans = safe_answer(q, a, "One-sentence answer with brief reasoning if possible.")
    records.append({"style": "reasoning", "question": q, "answer": reasoning_ans})

    short_ans = safe_answer(q, a, "Very short; keep numeric details; one sentence.")
    records.append({"style": "short", "question": q, "answer": short_ans})

    eli5_ans = safe_answer(q, a, "Explain simply in one sentence without losing numbers.")
    records.append({"style": "eli5", "question": q, "answer": eli5_ans})

    # Paraphrased questions + answers (re-answer using the same facts)
    for i, pq in enumerate(paraphrase_questions(q, a, n=3), start=1):
        pq_ans = safe_answer(pq, a, "Concise, factual; preserve numbers.")
        records.append({"style": f"paraphrase_{i}", "question": pq, "answer": pq_ans})

    # Analytical follow-up + answer
    follow_q = analytical_question(q, a)
    follow_a = safe_answer(follow_q, a, "Analytical but still one sentence; do not invent facts.")
    records.append({"style": "analytical", "question": follow_q, "answer": follow_a})

    return records

# =============================
# RUN AUGMENTATION
# =============================
extended_rows: List[Dict[str, str]] = []
for idx, row in df.iterrows():
    q, a = str(row["question"]).strip(), str(row["answer"]).strip()
    # Basic guard
    if not q or not a:
        continue
    try:
        extended_rows.extend(enrich_entry(q, a))
    except Exception as e:
        print(f"⚠️ Skipped row {idx} due to error: {e}")

extended_df = pd.DataFrame(extended_rows, columns=["style", "question", "answer"])
extended_df.to_csv(FINAL_CSV, index=False)

print(f"✅ Extended dataset saved to: {FINAL_CSV}")
print("📊 Final size:", len(extended_df))
print("\n🔍 Sample:")
print(extended_df.sample(min(8, len(extended_df))).to_string(index=False))


📊 Original dataset size: 68


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ Extended dataset saved to: /content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset/extended_finetune_dataset.csv
📊 Final size: 544

🔍 Sample:
       style                                                                                                                                                                                                                                                                                                 question                                                                                                                                         answer
paraphrase_3                                                                                                                                                                                                                                    Q: How are the segments of Nestle's revenue organized geographically?                                                                      

In [3]:
# import os
# import re
# import pandas as pd
# from pathlib import Path
# from typing import List, Dict
# import requests
# from bs4 import BeautifulSoup
# from transformers import pipeline

# # =============================
# # CONFIG
# # =============================
# OUTPUT_DIR = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset"
# MODEL_NAME = "google/flan-t5-small"
# Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
# FINAL_CSV = os.path.join(OUTPUT_DIR, "extended_finetune_dataset.csv")

# # =============================
# # LLM PIPELINE
# # =============================
# generator = pipeline("text2text-generation", model=MODEL_NAME, device=-1)
# GEN_KW = dict(max_new_tokens=96, num_beams=4, do_sample=False, repetition_penalty=1.05)

# def run_llm(prompt: str) -> str:
#     return generator(prompt, **GEN_KW)[0]["generated_text"].strip()

# # =============================
# # SCRAPING HELPERS
# # =============================
# def scrape_page_text(url: str, limit_paras: int = 3) -> str:
#     """Fetch visible text from webpage (for facts/definitions)."""
#     try:
#         resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
#         soup = BeautifulSoup(resp.text, "html.parser")
#         paras = [p.text.strip() for p in soup.find_all("p") if len(p.text.strip()) > 40]
#         return " ".join(paras[:limit_paras])
#     except Exception as e:
#         print(f"⚠️ Scrape failed: {e}")
#         return ""

# # =============================
# # QA GENERATORS
# # =============================
# def generate_fact_qa(fact_text: str, n: int = 5) -> List[Dict[str, str]]:
#     """Generate Q/A pairs grounded in factual text."""
#     prompt = f"""
#     You are a precise financial assistant.
#     FACT: {fact_text}
#     Generate {n} different Q/A pairs using ONLY this fact.
#     Keep questions simple and factual. Keep answers accurate, concise.
#     Format strictly as:
#     Q: ...
#     A: ...
#     """
#     out = run_llm(prompt)
#     qa_pairs = []
#     q, a = None, None
#     for line in out.splitlines():
#         if line.strip().startswith("Q:"):
#             q = line.replace("Q:", "").strip()
#         elif line.strip().startswith("A:"):
#             a = line.replace("A:", "").strip()
#             if q and a:
#                 qa_pairs.append({"style": "fact_based", "question": q, "answer": a})
#                 q, a = None, None
#     return qa_pairs

# def generate_term_qa(term: str, definition: str, n: int = 3) -> List[Dict[str, str]]:
#     """Generate QA pairs for financial terminology."""
#     prompt = f"""
#     Term: {term}
#     Definition: {definition}
#     Generate {n} Q/A pairs that explain or test understanding of this term.
#     Format strictly as:
#     Q: ...
#     A: ...
#     """
#     out = run_llm(prompt)
#     qa_pairs = []
#     q, a = None, None
#     for line in out.splitlines():
#         if line.strip().startswith("Q:"):
#             q = line.replace("Q:", "").strip()
#         elif line.strip().startswith("A:"):
#             a = line.replace("A:", "").strip()
#             if q and a:
#                 qa_pairs.append({"style": "term_based", "question": q, "answer": a})
#                 q, a = None, None
#     return qa_pairs

# # =============================
# # FACT SOURCES
# # =============================
# # ✅ Nestlé Annual Report / Fact Pages (scraped text chunks)
# nestle_urls = [
#     "https://www.nestle.com/investors/annual-report",
#     "https://www.nestle.com/media/pressreleases/allpressreleases/nestle-2023-full-year-results"
# ]

# # ✅ Basic finance terms (scraped from Investopedia)
# term_urls = {
#     "Revenue": "https://www.investopedia.com/terms/r/revenue.asp",
#     "Operating Profit Margin": "https://www.investopedia.com/terms/o/operating-margin.asp",
#     "Free Cash Flow": "https://www.investopedia.com/terms/f/freecashflow.asp",
#     "Earnings per Share (EPS)": "https://www.investopedia.com/terms/e/eps.asp",
#     "Dividend": "https://www.investopedia.com/terms/d/dividend.asp"
# }

# # =============================
# # DATASET BUILD
# # =============================
# extended_rows: List[Dict[str, str]] = []

# # 1) Scrape Nestlé pages and generate factual Q/A
# for url in nestle_urls:
#     fact_text = scrape_page_text(url)
#     if fact_text:
#         extended_rows.extend(generate_fact_qa(fact_text, n=10))

# # 2) Financial terminology Q/A
# for term, url in term_urls.items():
#     def_text = scrape_page_text(url)
#     if def_text:
#         extended_rows.extend(generate_term_qa(term, def_text, n=5))

# # =============================
# # CLEAN DUPLICATES
# # =============================
# df = pd.DataFrame(extended_rows)
# df.drop_duplicates(subset=["question"], inplace=True)

# # ensure we hit ~250+ entries (loop again if needed)
# while len(df) < 250:
#     # add more Q/A from finance terms to pad
#     for term, url in term_urls.items():
#         def_text = scrape_page_text(url)
#         if def_text:
#             df = pd.concat([df, pd.DataFrame(generate_term_qa(term, def_text, n=3))])
#     df.drop_duplicates(subset=["question"], inplace=True)

# # =============================
# # SAVE
# # =============================
# df.to_csv(FINAL_CSV, index=False)
# print(f"✅ Extended dataset saved: {FINAL_CSV}")
# print("📊 Final size:", len(df))
# print("\n🔍 Sample:")
# print(df.sample(10).to_string(index=False))


In [4]:
# ==========================================================
# 📊 Task 3.3 – Baseline Benchmarking (Pre-Fine-Tuning)
# ==========================================================
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

# ----------------------------------------------------------
# Load dataset
# ----------------------------------------------------------
CSV_PATH = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset/extended_finetune_dataset.csv"

df = pd.read_csv(CSV_PATH)

# Use only "question" and "answer" columns for Q/A benchmarking
test_df = df[["question", "answer"]].dropna().sample(10, random_state=42)  # pick 10 samples for baseline
print("Loaded test questions:", len(test_df))

# ----------------------------------------------------------
# Load base model + tokenizer
# ----------------------------------------------------------
model_name = "distilgpt2"   # you can change later to GPT-2 / Llama-2 small etc.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Similarity model
similarity_model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------------------------------------
# Benchmark function
# ----------------------------------------------------------
def run_benchmark(model, tokenizer, test_questions, true_answers):
    results = []
    for i, (question, true_ans) in enumerate(zip(test_questions, true_answers)):
        prompt = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        start_time = time.time()
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.1,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True
        )

        # Confidence from transition scores
        transition_scores = model.compute_transition_scores(
            outputs.sequences, outputs.scores, normalize_logits=True
        )
        avg_confidence = torch.exp(transition_scores[0]).mean().item()

        # Extract generated answer
        full_output = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
        generated_ans = full_output.split("Answer:")[-1].split("\n")[0].strip()

        inference_time = time.time() - start_time

        # Semantic similarity with ground truth
        embeddings = similarity_model.encode([generated_ans, true_ans], convert_to_tensor=True)
        cos_sim = util.cos_sim(embeddings[0], embeddings[1]).item()

        results.append({
            "question": question,
            "true_answer": true_ans,
            "generated_answer": generated_ans,
            "similarity_accuracy_score": round(cos_sim, 4),
            "confidence": round(avg_confidence, 4),
            "inference_time": round(inference_time, 4)
        })

        print(f"Processed {i+1}/{len(test_questions)} | Sim={cos_sim:.2f} | Time={inference_time:.2f}s")

    return results

# ----------------------------------------------------------
# Run Baseline Benchmark
# ----------------------------------------------------------
print("\n===================================================")
print("Running baseline benchmarking...\n")

test_questions = test_df["question"].tolist()
true_answers = test_df["answer"].tolist()

baseline_results = run_benchmark(model, tokenizer, test_questions, true_answers)

baseline_results_df = pd.DataFrame(baseline_results, columns=[
    "question", "true_answer", "generated_answer",
    "similarity_accuracy_score", "confidence", "inference_time"
])

print("\n===================================================")
print("✅ Baseline Benchmark Completed!")

# Save results
baseline_results_df.to_csv("baseline_benchmark_results.csv", index=False)


Loaded test questions: 10


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Running baseline benchmarking...

Processed 1/10 | Sim=0.50 | Time=24.30s
Processed 2/10 | Sim=0.15 | Time=28.28s
Processed 3/10 | Sim=0.15 | Time=25.12s
Processed 4/10 | Sim=0.10 | Time=14.39s
Processed 5/10 | Sim=0.86 | Time=14.24s
Processed 6/10 | Sim=0.46 | Time=14.11s
Processed 7/10 | Sim=0.63 | Time=17.38s
Processed 8/10 | Sim=0.10 | Time=13.95s
Processed 9/10 | Sim=0.01 | Time=13.89s
Processed 10/10 | Sim=0.06 | Time=16.59s

✅ Baseline Benchmark Completed!


In [5]:
baseline_results_df.head()

,question,true_answer,generated_answer,similarity_accuracy_score,confidence,inference_time
0,Q: What are the total number of sales transact...,"12,196 and 11,793 in 2024 and 2023 for custome...",The number of sales transactions that resulted...,0.5050,0.9994,24.3035
1,Q: By how much did financial income increase b...,"Financial income increased by CHF 74 million, ...","$1,000,000,000,000,000,000,000,000,000,000,000...",0.1502,0.9992,28.2778
2,Q: What did the 2024 other revenue of primaril...,The 2024 other revenue of primarily sale of ou...,The first thing we did was to get the most out...,0.1502,0.9888,25.1189
3,The segments are broken down by Zones North Am...,Not specified.,The segments are broken down by Zones North Am...,0.1020,1.0000,14.3875
4,Q: How does the trading operating profit show ...,The segment results (Trading operating profit)...,The trading operating profit shows its contrib...,0.8619,0.9972,14.2383


In [8]:
# ===========================================
# 🔹 Data Prep & Hyperparameter Setup
# ===========================================
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

# Load dataset directly from CSV you provided
CSV_PATH = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset/extended_finetune_dataset.csv"
df = pd.read_csv(CSV_PATH)

# Merge question + answer into a single training text
df["text"] = df.apply(lambda r: f"Q: {r['question']}\nA: {r['answer']}", axis=1)

# Convert to HuggingFace dataset
dataset = Dataset.from_pandas(df[["text"]])

# Tokenization helper
def prepare_batch(examples):
    return tokenizer(
        examples["text"],
        max_length=256,
        padding="max_length",
        truncation=True
    )

# Apply preprocessing
encoded_dataset = dataset.map(prepare_batch, batched=True)

# Collator for Causal LM (not MLM)
lm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training hyperparameters
train_settings = dict(
    lr=3e-5,
    batch=8,
    epochs=15,
    device_type="cuda" if torch.cuda.is_available() else "cpu",
    base_model=model_name,
    optimizer_type="AdamW",
    decay=0.03,
    warmup=100,
    grad_accum=4
)

# Print them nicely
print("\n⚙️ Training Parameters")
print("-" * 50)
for param, value in train_settings.items():
    print(f"{param:15} → {value}")


Map:   0%|          | 0/544 [00:00<?, ? examples/s]


⚙️ Training Parameters
--------------------------------------------------
lr              → 3e-05
batch           → 8
epochs          → 15
device_type     → cpu
base_model      → distilgpt2
optimizer_type  → AdamW
decay           → 0.03
warmup          → 100
grad_accum      → 4


In [ ]:
import time
import torch
from transformers import TrainingArguments, Trainer

# ==============================
# Training Arguments
# ==============================
train_args = TrainingArguments(
    output_dir="./finetuned_nestle_gpt2",
    save_strategy="epoch",
    num_train_epochs=train_settings["epochs"],
    per_device_train_batch_size=train_settings["batch"],
    learning_rate=train_settings["lr"],
    weight_decay=train_settings["decay"],
    logging_dir="./logs",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=train_settings["grad_accum"],
    report_to="tensorboard",
    disable_tqdm=False
)

# ==============================
# Trainer Setup
# ==============================
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=encoded_dataset,
    data_collator=lm_collator
)

# ==============================
# Run Fine-Tuning
# ==============================
print("\n🚀 Starting supervised instruction fine-tuning...")
t0 = time.time()
trainer.train()
print(f"\n✅ Fine-tuning finished in {(time.time()-t0)/60:.2f} minutes")

# Save final model + tokenizer
save_dir = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/extended_dataset/finetuned_nestle_gpt2/"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"📦 Model + tokenizer saved at: {save_dir}")



🚀 Starting supervised instruction fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.881800
20,2.304700
30,2.050800
40,1.788300
50,1.648000
60,1.463800
70,1.349700
80,1.264900


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument i

In [ ]:
# nestle_guardrails.py
# PEP-8 compliant, extended guardrails for a Nestlé financial Q&A assistant.

from __future__ import annotations
import re
from dataclasses import dataclass
from typing import Tuple, Optional, Dict, List


@dataclass(frozen=True)
class GuardrailHit:
    """Structured result for a blocked query."""
    category: str
    reason: str
    message: str


class NestleInputGuard:
    """
    Input-side guardrails specialized for Nestlé financial Q&A.

    Categories reflect Nestlé policies and finance constraints:
    - forward_looking: block predictions, stock tips, or guidance.
    - insider_info: reject requests for non-public or confidential data.
    - pii_request: block attempts to extract personal/sensitive data.
    - illegal_activity: prevent violent/illegal/criminal topics.
    - off_topic: nudge back if user asks unrelated/non-financial queries.
    - vague_query: ask user to rephrase unclear questions.
    - financial_scope: enforce focus on *financial* metrics, not recipes, history, or HR.
    """

    def __init__(self) -> None:
        # Forward-looking investment guidance
        self.forward_looking = {
            "patterns": [
                r"\bforecast\b", r"\bprojection(s)?\b", r"\bguidance\b",
                r"\bnext\s+(quarter|year|q[1-4]|fy)\b",
                r"\b(price|stock|share)\s*(target|prediction|will\s+go)\b",
                r"\bshould\s+(i|we)\s+(buy|sell|hold)\b",
                r"\bexpected\s+(sales|revenue|eps|margin)\b",
            ],
            "message": "🚫 I cannot provide forward-looking guidance or stock tips. I can summarize past published results instead.",
            "reason": "Forward-looking or investment advice."
        }

        # Insider/confidential data
        self.insider_info = {
            "patterns": [
                r"\binsider\b", r"\bnon[-\s]?public\b", r"\bconfidential\b",
                r"\bleak\b", r"\bunreleased\b", r"\bearnings\s*(leak|before\s+release)\b",
            ],
            "message": "🚫 I cannot provide non-public or confidential information. I only use published, official data.",
            "reason": "Request for insider or confidential information."
        }

        # Personal identifiable info
        self.pii_request = {
            "patterns": [
                r"\bsocial\s*security\b", r"\bcredit\s*card\b", r"\bpassword\b",
                r"\bprivate\s*key\b", r"\baddress\b", r"\bphone\s*number\b",
                r"\bemail\b"
            ],
            "message": "🚫 I cannot provide or process personal or sensitive data.",
            "reason": "Solicitation of personal/sensitive information."
        }

        # Illegal/violent content
        self.illegal_activity = {
            "patterns": [
                r"\bbomb\b", r"\battack\b", r"\bkill\b", r"\bshoot\b",
                r"\bterror\b", r"\bmurder\b", r"\bfraud\b",
                r"\binsider\s+trading\b", r"\bscam\b", r"\blaunder\s+money\b"
            ],
            "message": "🚫 I cannot help with harmful, illegal, or violent requests.",
            "reason": "Illegal/violent content."
        }

        # Off-topic queries
        self.off_topic = {
            "patterns": [
                r"\b(recipe|cook|chocolate\s*cake|kitkat)\b",
                r"\bmovie\b", r"\bsport(s)?\b", r"\bweather\b", r"\bcelebrity\b"
            ],
            "message": "ℹ️ I’m focused on Nestlé’s financial performance. Try asking about revenue, debt, margins, or segments.",
            "reason": "Out of Nestlé finance scope."
        }

        # Vague queries
        self.vague_query = {
            "patterns": [
                r"\b(tell\s+me\s+something)\b",
                r"\b(anything|everything)\b",
                r"\bwhat\s+about\s+nestl[eé]\b"
            ],
            "message": "🤔 Could you rephrase your question to be more specific? For example: 'What was Nestlé’s revenue in 2023?'",
            "reason": "Vague/underspecified question."
        }

        # Explicit financial scope enforcement
        self.financial_scope = {
            "patterns": [
                r"\b(hr|hiring|job|employee|salary)\b",
                r"\bmarketing\s+campaign\b",
                r"\bhistory\s+of\s+nestl[eé]\b",
                r"\bproduct\s+ingredients\b"
            ],
            "message": "ℹ️ This assistant only answers questions about Nestlé’s financial metrics, reports, and performance.",
            "reason": "Non-financial scope detected."
        }

        # Compile regex once
        self._compiled: Dict[str, List[re.Pattern]] = {
            "forward_looking": [re.compile(p, re.IGNORECASE) for p in self.forward_looking["patterns"]],
            "insider_info": [re.compile(p, re.IGNORECASE) for p in self.insider_info["patterns"]],
            "pii_request": [re.compile(p, re.IGNORECASE) for p in self.pii_request["patterns"]],
            "illegal_activity": [re.compile(p, re.IGNORECASE) for p in self.illegal_activity["patterns"]],
            "off_topic": [re.compile(p, re.IGNORECASE) for p in self.off_topic["patterns"]],
            "vague_query": [re.compile(p, re.IGNORECASE) for p in self.vague_query["patterns"]],
            "financial_scope": [re.compile(p, re.IGNORECASE) for p in self.financial_scope["patterns"]],
        }

    def assess(self, query: str) -> Tuple[bool, Optional[GuardrailHit]]:
        """Return (allowed, GuardrailHit|None)."""
        checks = [
            ("illegal_activity", self.illegal_activity),
            ("pii_request", self.pii_request),
            ("insider_info", self.insider_info),
            ("forward_looking", self.forward_looking),
            ("financial_scope", self.financial_scope),
            ("off_topic", self.off_topic),
            ("vague_query", self.vague_query),
        ]

        for key, config in checks:
            for rx in self._compiled[key]:
                if rx.search(query):
                    return False, GuardrailHit(
                        category=key,
                        reason=config["reason"],
                        message=config["message"],
                    )
        return True, None


class SimpleOutputGuard:
    """
    Output-side guardrail:
    - removes repeated Q:/A: echoes,
    - trims long rambles,
    - enforces single concise factual sentence,
    - flags uncertainty.
    """

    UNCERTAIN_TOKENS = re.compile(r"\b(maybe|perhaps|i\s*think|not\s*sure|guess)\b", re.IGNORECASE)

    def clean(self, text: str) -> str:
        """Remove echoes, duplicates, and keep concise answers."""
        # Remove repeated Context/Q:/A:
        text = re.sub(r"(context:.*?\n+)", "", text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r"(?:Q:.*?\n)?\s*A:\s*", "", text, flags=re.IGNORECASE)

        # Collapse whitespace
        text = re.sub(r"\s+", " ", text).strip()

        # If multiple sentences, keep only the first 2 to avoid rambles
        sentences = re.split(r"(?<=[.!?])\s+", text)
        if len(sentences) > 2:
            text = " ".join(sentences[:2])

        return text

    def flag_uncertain(self, text: str) -> bool:
        """True if answer contains hedgy/uncertain wording."""
        return bool(self.UNCERTAIN_TOKENS.search(text))


Test guardrail

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ======================================
# Load fine-tuned Nestlé GPT-2 model
# ======================================
SAVE_DIR = "./finetuned_nestle_gpt2/final"

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForCausalLM.from_pretrained(SAVE_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# ======================================
# Guardrails (import your guard classes)
# ======================================

guard_in = NestleInputGuard()
guard_out = SimpleOutputGuard()

# ======================================
# Guarded Generation Function
# ======================================
def guarded_generate(query: str, max_new_tokens: int = 128):
    """Run guardrails around fine-tuned LLM output"""
    # --- Input Guard ---
    ok, hit = guard_in.assess(query)
    if not ok:
        return hit.message  # block unsafe query early

    # --- Model Inference ---
    prompt = f"Question: {query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic answers
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer after "Answer:" if exists
    if "Answer:" in raw_answer:
        raw_answer = raw_answer.split("Answer:")[-1].strip()

    # --- Output Guard ---
    clean_answer = guard_out.clean(raw_answer)
    if guard_out.flag_uncertain(clean_answer):
        clean_answer += "\n\n⚠️ This answer may be uncertain. Please recheck."

    return clean_answer

# ======================================
# 🔍 Test
# ======================================
user_query = "what kind of people netle hiring?"
print("User:", user_query)
print("Bot:", guarded_generate(user_query))

In [ ]:
import gradio as gr
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# Load Fine-Tuned Model
# =========================
MODEL_DIR = "./finetuned_nestle_gpt2/final"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

guard_in = NestleInputGuard()
guard_out = SimpleOutputGuard()

# =========================
# Answer Cleaning
# =========================
def clean_answer(raw_text: str) -> str:
    # Remove "Not specified" boilerplate
    text = raw_text.replace("Answer: Not specified.", "").strip()

    # Deduplicate repeated sentences
    parts = text.split(". ")
    deduped = []
    for p in parts:
        if p not in deduped:
            deduped.append(p)
    text = ". ".join(deduped)

    # If answer is still empty, fallback
    if not text or text.lower() in ["not specified", "n/a"]:
        text = "This information is not available in Nestlé's annual reports."

    return text.strip()

# =========================
# Inference Pipeline
# =========================
def answer_with_finetuned(query: str):
    # 1. Guardrail check
    allowed, hit = guard_in.assess(query)
    if not allowed:
        return hit.message, 0.0, "Guardrail", 0.0

    # 2. Generate from LLM
    prompt = f"Based only on Nestlé financial reports, provide a clear factual answer.\nQuestion: {query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )
    inference_time = round(time.time() - start, 3)

    raw_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the part after "Answer:"
    if "Answer:" in raw_text:
        raw_text = raw_text.split("Answer:")[-1].strip()

    # 3. Output guardrail cleaning
    final_answer = guard_out.clean(raw_text)
    final_answer = clean_answer(final_answer)

    # Confidence is not directly available for GPT-2 → we give neutral 1.0
    return final_answer, 1.0, "Fine-tuned LLM", inference_time

# =========================
# Gradio UI
# =========================
with gr.Blocks() as demo:
    gr.Markdown("## 🟢 Nestlé Financial Q&A (Fine-tuned Model with Guardrails)")

    with gr.Row():
        with gr.Column(scale=2):
            query = gr.Textbox(label="Ask a financial question about Nestlé")
            btn = gr.Button("Get Answer")

        with gr.Column(scale=3):
            answer = gr.Textbox(label="Answer", interactive=False)
            confidence = gr.Textbox(label="Confidence", interactive=False)
            method = gr.Textbox(label="Method", interactive=False)
            inference_time = gr.Textbox(label="Inference Time (s)", interactive=False)

    btn.click(
        fn=answer_with_finetuned,
        inputs=[query],
        outputs=[answer, confidence, method, inference_time],
    )

demo.launch()
